In [0]:
df = spark.read.csv("/Volumes/workspace/default/mandi_price_data/", header=True, inferSchema=True)
df.show(5)
df.printSchema()


+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|Arrival_Date|Commodity|Commodity_Code|District|Grade|  Market|Max_Price|Min_Price|Modal_Price|      State|Variety|
+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|  2024-08-10|    Onion|            23|  Nashik|  FAQ|Chandvad|   3433.0|   1500.0|     3350.0|Maharashtra|  Other|
|  2024-07-24|    Onion|            23|  Nashik|  FAQ|Chandvad|   2870.0|   1440.0|     2700.0|Maharashtra|  Other|
|  2024-07-26|    Onion|            23|  Nashik|  FAQ|Chandvad|   2880.0|   1300.0|     2700.0|Maharashtra|  Other|
|  2024-07-01|    Onion|            23|  Nashik|  FAQ|Chandvad|   3400.0|   1701.0|     3000.0|Maharashtra|  Other|
|  2024-07-06|    Onion|            23|  Nashik|  FAQ|Chandvad|   3337.0|   1201.0|     2980.0|Maharashtra|  Other|
+------------+---------+--------------+--------+-----+--------+---------

In [0]:
df.printSchema()

root
 |-- Arrival_Date: date (nullable = true)
 |-- Commodity: string (nullable = true)
 |-- Commodity_Code: integer (nullable = true)
 |-- District: string (nullable = true)
 |-- Grade: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Max_Price: double (nullable = true)
 |-- Min_Price: double (nullable = true)
 |-- Modal_Price: double (nullable = true)
 |-- State: string (nullable = true)
 |-- Variety: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, trim, upper

# Step 1: remove rows with missing or zero price (using Modal_Price, Min_Price, Max_Price)
# Add this to your Step 1 market_avg calculation, or better, add it back in your original df_clean cleaning step
df_clean_v2 = df_clean.filter(
    (col("Modal_Price") > 0) & (col("Modal_Price") < 20000) &
    (col("Min_Price") > 0) & (col("Min_Price") < 20000) &
    (col("Max_Price") > 0) & (col("Max_Price") < 20000)
)

print("Rows before outlier filter:", df_clean.count())
print("Rows after outlier filter:", df_clean_v2.count())

# Step 2: standardize text columns (remove extra spaces, make consistent case)
df_clean = df_clean.withColumn("State", trim(upper(col("State")))) \
                    .withColumn("District", trim(upper(col("District")))) \
                    .withColumn("Market", trim(upper(col("Market")))) \
                    .withColumn("Commodity", trim(upper(col("Commodity")))) \
                    .withColumn("Variety", trim(upper(col("Variety"))))

# Step 3: remove exact duplicate rows
df_clean = df_clean.dropDuplicates()

# Check results
print("Rows before cleaning:", df_clean.count())
print("Rows after cleaning:", df_clean.count())
df_clean.show(5)

Rows before outlier filter: 109337
Rows after outlier filter: 109324
Rows before cleaning: 109337
Rows after cleaning: 109337
+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|Arrival_Date|Commodity|Commodity_Code|District|Grade|  Market|Max_Price|Min_Price|Modal_Price|      State|Variety|
+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|  2024-01-17|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1691.0|    600.0|     1460.0|MAHARASHTRA|    RED|
|  2024-02-12|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1381.0|    500.0|     1150.0|MAHARASHTRA|    RED|
|  2025-01-03|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   2781.0|   1212.0|     2200.0|MAHARASHTRA|    RED|
|  2023-04-26|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1025.0|    350.0|      600.0|MAHARASHTRA|  OTHER|
|  2023-07-14|    ONION|            23|  NASHIK|  FAQ|CHANDVAD

In [0]:
df_clean.show(5)

+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|Arrival_Date|Commodity|Commodity_Code|District|Grade|  Market|Max_Price|Min_Price|Modal_Price|      State|Variety|
+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|  2024-01-17|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1691.0|    600.0|     1460.0|MAHARASHTRA|    RED|
|  2024-02-12|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1381.0|    500.0|     1150.0|MAHARASHTRA|    RED|
|  2025-01-03|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   2781.0|   1212.0|     2200.0|MAHARASHTRA|    RED|
|  2023-04-26|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1025.0|    350.0|      600.0|MAHARASHTRA|  OTHER|
|  2023-07-14|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1650.0|    290.0|     1100.0|MAHARASHTRA|  OTHER|
+------------+---------+--------------+--------+-----+--------+---------

In [0]:
df_clean_v2.write.mode("overwrite").saveAsTable("mandi_prices_clean")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5242458494240229>, line 1
----> 1 df_clean_v2.write.mode("overwrite").saveAsTable("mandi_prices_clean")

NameError: name 'df_clean_v2' is not defined

In [0]:
spark.sql("SELECT COUNT(*) FROM mandi_prices_clean").show()

+--------+
|COUNT(*)|
+--------+
|  109337|
+--------+



In [0]:
spark.sql("""
SELECT Commodity, COUNT(*) as row_count
FROM mandi_prices_clean
GROUP BY Commodity
ORDER BY row_count DESC
""").show()

+--------------------+---------+
|           Commodity|row_count|
+--------------------+---------+
|               ONION|    38486|
|              POTATO|    22776|
|              TOMATO|    21318|
|               WHEAT|    20505|
|BENGAL GRAM(GRAM)...|     6252|
+--------------------+---------+



In [0]:
%sql
SELECT Market, District, State, Commodity, Modal_Price, Arrival_Date
FROM mandi_prices_clean
WHERE Arrival_Date = (SELECT MAX(Arrival_Date) FROM mandi_prices_clean)
ORDER BY Modal_Price DESC
LIMIT 10


Market,District,State,Commodity,Modal_Price,Arrival_Date
AKOLA APMC,AKOLA,MAHARASHTRA,BENGAL GRAM(GRAM)(WHOLE),5120.0,2025-12-31
INDORE APMC,INDORE,MADHYA PRADESH,BENGAL GRAM(GRAM)(WHOLE),5000.0,2025-12-31
INDORE APMC,INDORE,MADHYA PRADESH,BENGAL GRAM(GRAM)(WHOLE),5000.0,2025-12-31
MHOW APMC,INDORE,MADHYA PRADESH,BENGAL GRAM(GRAM)(WHOLE),4990.0,2025-12-31
MURTIZAPUR APMC,AKOLA,MAHARASHTRA,BENGAL GRAM(GRAM)(WHOLE),4890.0,2025-12-31
"BINNY MILL (F&V), BANGALORE APMC",BANGALORE,KARNATAKA,TOMATO,4700.0,2025-12-31
MADANAPALLI APMC,CHITTOR,ANDHRA PRADESH,TOMATO,4700.0,2025-12-31
LATUR APMC,LATUR,MAHARASHTRA,BENGAL GRAM(GRAM)(WHOLE),4600.0,2025-12-31
KALIKIRI APMC,CHITTOR,ANDHRA PRADESH,TOMATO,4500.0,2025-12-31
RAMANAGARA APMC,BANGALORE,KARNATAKA,TOMATO,4500.0,2025-12-31


In [0]:
%sql
SELECT 
  Commodity, State, Market, Arrival_Date, Modal_Price,
  ROUND(AVG(Modal_Price) OVER (
    PARTITION BY Commodity, Market 
    ORDER BY Arrival_Date 
    ROWS BETWEEN 29 PRECEDING AND CURRENT ROW
  ), 2) as rolling_30day_avg
FROM mandi_prices_clean
WHERE Commodity = 'ONION' AND Market = 'CHANDVAD'
ORDER BY Arrival_Date

Commodity,State,Market,Arrival_Date,Modal_Price,rolling_30day_avg
ONION,MAHARASHTRA,CHANDVAD,2023-01-02,1300.0,1300.0
ONION,MAHARASHTRA,CHANDVAD,2023-01-02,1200.0,1250.0
ONION,MAHARASHTRA,CHANDVAD,2023-01-03,1350.0,1283.33
ONION,MAHARASHTRA,CHANDVAD,2023-01-04,1400.0,1312.5
ONION,MAHARASHTRA,CHANDVAD,2023-01-06,1370.0,1324.0
ONION,MAHARASHTRA,CHANDVAD,2023-01-07,1480.0,1350.0
ONION,MAHARASHTRA,CHANDVAD,2023-01-10,1300.0,1342.86
ONION,MAHARASHTRA,CHANDVAD,2023-01-11,1270.0,1333.75
ONION,MAHARASHTRA,CHANDVAD,2023-01-12,1270.0,1326.67
ONION,MAHARASHTRA,CHANDVAD,2023-01-13,1250.0,1319.0


In [0]:
%sql
SELECT 
  Commodity, State, Market, 
  YEAR(Arrival_Date) as year,
  MONTH(Arrival_Date) as month,
  ROUND(AVG(Modal_Price), 2) as monthly_avg_price,
  ROUND(LAG(AVG(Modal_Price)) OVER (
    PARTITION BY Commodity, Market, MONTH(Arrival_Date)
    ORDER BY YEAR(Arrival_Date)
  ), 2) as prev_year_same_month_price
FROM mandi_prices_clean
WHERE Commodity = 'ONION' AND Market = 'CHANDVAD'
GROUP BY Commodity, State, Market, YEAR(Arrival_Date), MONTH(Arrival_Date)
ORDER BY month, year

Commodity,State,Market,year,month,monthly_avg_price,prev_year_same_month_price
ONION,MAHARASHTRA,CHANDVAD,2023,1,1246.84,null
ONION,MAHARASHTRA,CHANDVAD,2024,1,1531.82,1246.84
ONION,MAHARASHTRA,CHANDVAD,2025,1,2046.54,1531.82
ONION,MAHARASHTRA,CHANDVAD,2023,2,695.24,null
ONION,MAHARASHTRA,CHANDVAD,2024,2,1407.05,695.24
ONION,MAHARASHTRA,CHANDVAD,2025,2,2327.65,1407.05
ONION,MAHARASHTRA,CHANDVAD,2023,3,761.47,null
ONION,MAHARASHTRA,CHANDVAD,2024,3,1482.56,761.47
ONION,MAHARASHTRA,CHANDVAD,2025,3,1502.22,1482.56
ONION,MAHARASHTRA,CHANDVAD,2023,4,559.29,null


In [0]:
%sql
SELECT 
  Commodity, State,
  YEAR(Arrival_Date) as year,
  MONTH(Arrival_Date) as month,
  ROUND(AVG(Modal_Price), 2) as monthly_avg_price,
  RANK() OVER (
    PARTITION BY Commodity, YEAR(Arrival_Date), MONTH(Arrival_Date)
    ORDER BY AVG(Modal_Price) DESC
  ) as price_rank
FROM mandi_prices_clean
WHERE Commodity = 'ONION'
GROUP BY Commodity, State, YEAR(Arrival_Date), MONTH(Arrival_Date)
ORDER BY year, month, price_rank

Commodity,State,year,month,monthly_avg_price,price_rank
ONION,KARNATAKA,2023,1,1254.03,1
ONION,MAHARASHTRA,2023,1,1210.76,2
ONION,MADHYA PRADESH,2023,1,960.61,3
ONION,KARNATAKA,2023,2,976.86,1
ONION,MAHARASHTRA,2023,2,946.1,2
ONION,MADHYA PRADESH,2023,2,869.17,3
ONION,KARNATAKA,2023,3,964.43,1
ONION,MADHYA PRADESH,2023,3,866.32,2
ONION,MAHARASHTRA,2023,3,785.4,3
ONION,KARNATAKA,2023,4,869.51,1


In [0]:
df_clean = spark.table("mandi_prices_clean")
df_clean.show(5)

+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|Arrival_Date|Commodity|Commodity_Code|District|Grade|  Market|Max_Price|Min_Price|Modal_Price|      State|Variety|
+------------+---------+--------------+--------+-----+--------+---------+---------+-----------+-----------+-------+
|  2024-01-17|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1691.0|    600.0|     1460.0|MAHARASHTRA|    RED|
|  2024-02-12|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1381.0|    500.0|     1150.0|MAHARASHTRA|    RED|
|  2025-01-03|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   2781.0|   1212.0|     2200.0|MAHARASHTRA|    RED|
|  2023-04-26|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1025.0|    350.0|      600.0|MAHARASHTRA|  OTHER|
|  2023-07-14|    ONION|            23|  NASHIK|  FAQ|CHANDVAD|   1650.0|    290.0|     1100.0|MAHARASHTRA|  OTHER|
+------------+---------+--------------+--------+-----+--------+---------

In [0]:
from pyspark.sql.functions import col, datediff, lit, abs as spark_abs

msp_date = "2024-10-16"

wheat_before = df_clean.filter(
    (col("Commodity") == "WHEAT") &
    (datediff(lit(msp_date), col("Arrival_Date")) > 0) &
    (datediff(lit(msp_date), col("Arrival_Date")) <= 60)
)

wheat_after = df_clean.filter(
    (col("Commodity") == "WHEAT") &
    (datediff(col("Arrival_Date"), lit(msp_date)) > 0) &
    (datediff(col("Arrival_Date"), lit(msp_date)) <= 60)
)

print("Rows before MSP announcement:", wheat_before.count())
print("Rows after MSP announcement:", wheat_after.count())

Rows before MSP announcement: 1160
Rows after MSP announcement: 1116


In [0]:
from scipy import stats
import numpy as np

# Convert Spark data to plain Python lists so we can run statistics on them
before_prices = [row.Modal_Price for row in wheat_before.select("Modal_Price").collect()]
after_prices = [row.Modal_Price for row in wheat_after.select("Modal_Price").collect()]

before_prices = np.array(before_prices)
after_prices = np.array(after_prices)

# Basic stats first, in plain numbers
print("BEFORE MSP announcement:")
print("  Average price:", round(np.mean(before_prices), 2))
print("  Std deviation (volatility):", round(np.std(before_prices), 2))

print("\nAFTER MSP announcement:")
print("  Average price:", round(np.mean(after_prices), 2))
print("  Std deviation (volatility):", round(np.std(after_prices), 2))

# F-test: compares whether the VARIANCE (volatility) is significantly different
f_stat = np.var(before_prices, ddof=1) / np.var(after_prices, ddof=1)
df1 = len(before_prices) - 1
df2 = len(after_prices) - 1
p_value_f = 2 * min(stats.f.cdf(f_stat, df1, df2), 1 - stats.f.cdf(f_stat, df1, df2))

print("\n--- F-test (volatility comparison) ---")
print("F-statistic:", round(f_stat, 4))
print("p-value:", round(p_value_f, 4))

# t-test: compares whether the AVERAGE price is significantly different (secondary check)
t_stat, p_value_t = stats.ttest_ind(before_prices, after_prices)
print("\n--- t-test (average price comparison) ---")
print("t-statistic:", round(t_stat, 4))
print("p-value:", round(p_value_t, 4))

BEFORE MSP announcement:
  Average price: 2667.66
  Std deviation (volatility): 184.42

AFTER MSP announcement:
  Average price: 2801.85
  Std deviation (volatility): 152.67

--- F-test (volatility comparison) ---
F-statistic: 1.459
p-value: 0.0

--- t-test (average price comparison) ---
t-statistic: -18.8619
p-value: 0.0


In [0]:
from pyspark.sql.functions import avg, round, count, max as spark_max, min as spark_min

market_avg = df_clean.groupBy("Commodity", "District", "Market") \
    .agg(
        round(avg("Modal_Price"), 2).alias("avg_price"),
        count("*").alias("num_records")
    ) \
    .filter(col("num_records") >= 10)  # only markets with enough data to trust the average

market_avg.show(20)

+---------+--------+--------------------+----------+-----------+
|Commodity|District|              Market| avg_price|num_records|
+---------+--------+--------------------+----------+-----------+
|    ONION|  NASHIK|            CHANDVAD|   1926.29|        838|
|    ONION|  NASHIK|       CHANDVAD APMC|   1594.09|         57|
|    ONION|  NASHIK|              DEVALA|   1999.59|        709|
|    ONION|  NASHIK|         DEVALA APMC|   1422.38|         61|
|    ONION|  NASHIK|             DINDORI|   1407.36|        241|
|    ONION|  NASHIK|       DINDORI(VANI)|   1864.93|        457|
|    ONION|  NASHIK|  DINDORI(VANI) APMC|    1695.8|         15|
|    ONION|  NASHIK|              KALVAN|1603304.65|        573|
|    ONION|  NASHIK|         KALVAN APMC|   1161.62|         45|
|    ONION|  NASHIK|           LASALGAON|   2032.64|        930|
|    ONION|  NASHIK|      LASALGAON APMC|   1800.58|         64|
|    ONION|  NASHIK|   LASALGAON(NIPHAD)|   2034.08|        891|
|    ONION|  NASHIK|LASAL

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, col

window_spec = Window.partitionBy("Commodity", "District").orderBy(col("avg_price").desc())

ranked_markets = market_avg.withColumn("rank", rank().over(window_spec))

best_markets = ranked_markets.filter(col("rank") == 1)
best_markets.show(20)

+--------------------+------------+--------------------+----------+-----------+----+
|           Commodity|    District|              Market| avg_price|num_records|rank|
+--------------------+------------+--------------------+----------+-----------+----+
|BENGAL GRAM(GRAM)...|       AKOLA|                AKOT|   6261.03|         74|   1|
|BENGAL GRAM(GRAM)...|      INDORE|         SANWER APMC|   8253.51|         10|   1|
|BENGAL GRAM(GRAM)...|       LATUR|               LATUR|   5678.66|        460|   1|
|               ONION|  AHMEDNAGAR|               AKOLE|   2010.16|        251|   1|
|               ONION|   BANGALORE|          RAMANAGARA|   3416.67|        108|   1|
|               ONION|     DHARWAD|    HUBLI (AMARAGOL)|   1323.01|       1648|   1|
|               ONION|      INDORE|              INDORE|   1825.19|       1433|   1|
|               ONION|      NASHIK|              KALVAN|1603304.65|        573|   1|
|               ONION|     NEEMUCH|             NEEMUCH|   1525.0

In [0]:
market_avg = df_clean_v2.groupBy("Commodity", "District", "Market") \
    .agg(
        round(avg("Modal_Price"), 2).alias("avg_price"),
        count("*").alias("num_records")
    ) \
    .filter(col("num_records") >= 10)

window_spec = Window.partitionBy("Commodity", "District").orderBy(col("avg_price").desc())
ranked_markets = market_avg.withColumn("rank", rank().over(window_spec))
best_markets = ranked_markets.filter(col("rank") == 1)

worst_or_other_markets = ranked_markets.filter(col("rank") > 1) \
    .groupBy("Commodity", "District") \
    .agg(round(avg("avg_price"), 2).alias("avg_of_other_markets"))

advisory = best_markets.join(worst_or_other_markets, on=["Commodity", "District"], how="inner") \
    .withColumn("price_gap_per_quintal", round(col("avg_price") - col("avg_of_other_markets"), 2)) \
    .select("Commodity", "District", "Market", "avg_price", "avg_of_other_markets", "price_gap_per_quintal") \
    .orderBy(col("price_gap_per_quintal").desc())

advisory.show(20, truncate=False)

+------------------------+----------+--------------------------------------------------+---------+--------------------+---------------------+
|Commodity               |District  |Market                                            |avg_price|avg_of_other_markets|price_gap_per_quintal|
+------------------------+----------+--------------------------------------------------+---------+--------------------+---------------------+
|BENGAL GRAM(GRAM)(WHOLE)|INDORE    |SANWER APMC                                       |8253.51  |5764.64             |2488.87              |
|ONION                   |PUNE      |SHIRUR                                            |3413.79  |1583.71             |1830.08              |
|ONION                   |BANGALORE |RAMANAGARA                                        |3416.67  |1903.64             |1513.03              |
|TOMATO                  |KOLAR     |GOWRIBIDANOOR APMC                                |3268.75  |1798.61             |1470.14              |
|TOMAT

In [0]:
from pyspark.sql.functions import col

# Reload the base clean table
df_clean = spark.table("mandi_prices_clean")

# Re-apply the outlier filter
df_clean_v2 = df_clean.filter(
    (col("Modal_Price") > 0) & (col("Modal_Price") < 20000) &
    (col("Min_Price") > 0) & (col("Min_Price") < 20000) &
    (col("Max_Price") > 0) & (col("Max_Price") < 20000)
)

print("Rows before:", df_clean.count())
print("Rows after:", df_clean_v2.count())

# Save it back, overwriting the table
df_clean_v2.write.mode("overwrite").saveAsTable("mandi_prices_clean")

# Confirm final saved count
spark.sql("SELECT COUNT(*) FROM mandi_prices_clean").show()

Rows before: 109337
Rows after: 109324
+--------+
|COUNT(*)|
+--------+
|  109324|
+--------+

